# CRIVO — Fase 6b: conversar com falas de pessoas (A100)

Parte da base crescida da fase 6a (46M, 24 camadas, contexto de 1024 tokens, conferida pelo SHA do `resumo_fase6a.json`) e ensina o CRIVO a conversar. **Ninguém escreve as falas dele:** os alvos de treino são respostas escritas por pessoas e textos do acervo (`experimentos/fase6b_conversa_humana/gerar_fase6b.py`):

1. **OASST2 em português** (Apache-2.0): conversas inteiras escritas por voluntários; ficam fora as mensagens sintéticas, apagadas ou reprovadas na revisão;
2. **legendas pt-BR** (OpenSubtitles v2018, OPUS; uso não comercial, crédito a opensubtitles.org): falas seguidas de filmes e séries;
3. **acervo do CRIVO**: a pergunta "me fala sobre X" e, como resposta, os fatos do próprio acervo (ensina a usar o que recebe no contexto).

O contexto traz `quem` (o resumo do `IDENTIDADE.md`: ele lê quem é e se apresenta do jeito dele) e `lembrancas` (o que a pessoa disse antes, para ele aprender a lembrar). Um quarto de cada lote continua sendo leitura (web, corpus próprio e o `IDENTIDADE.md` inteiro), para ele não esquecer o que leu.

**Quanto treina:** 4 voltas no OASST2, o acervo e um tanto de legendas novas a cada volta; o número de passos sai do tamanho real dos dados (impresso na célula 3). Fica a versão com a menor perda nas conversas separadas.

**Medidas:** perda nas conversas separadas (antes e depois), perda de leitura (para ver se esqueceu), e respostas dele, sem retoque, às frases reais do criador, a "quem é você?", a perguntas com fatos e a testes de lembrança.

> **A100 → Executar tudo.** Precisa da fase 6a terminada na mesma conta (`CRIVO/transformer-crivo-46m/resumo_fase6a.json`). Pode parar e continuar: "Executar tudo" de novo continua do checkpoint. Saída: `CRIVO/transformer-crivo-46m-conversa/`; o resumo pequeno é `resumo_fase6b.json`.

In [ ]:
# Célula 0 — GPU, Drive e a base crescida da fase 6a
import os, sys, json, shutil, subprocess, hashlib, time, math, random, glob, re, gzip
from pathlib import Path
from datetime import datetime
from collections import Counter
import numpy as np
import torch
import torch.nn.functional as F

if not torch.cuda.is_available():
    raise SystemExit("ERRO: GPU não disponível. Ambiente de execução → Alterar tipo de ambiente → A100.")
GPU = torch.cuda.get_device_name(0)
print("GPU:", GPU, "| PyTorch", torch.__version__, flush=True)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True
DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

def sha256(p):
    h = hashlib.sha256()
    with open(p, "rb") as f:
        for b in iter(lambda: f.read(1 << 20), b""):
            h.update(b)
    return h.hexdigest()

MEU = Path("/content/drive/MyDrive")
candidatas = [MEU / "CRIVO"] + sorted(p for p in MEU.iterdir() if p.is_dir() and p.name != "CRIVO")
CRIVO = next((p for p in candidatas if (p / "transformer-crivo-46m" / "resumo_fase6a.json").is_file()), None)
if CRIVO is None:
    raise SystemExit("ERRO: não achei CRIVO/transformer-crivo-46m/resumo_fase6a.json. Rode na conta onde a fase 6a treinou (hr87676620).")
R6A = json.loads((CRIVO / "transformer-crivo-46m" / "resumo_fase6a.json").read_text())
if not R6A.get("terminou") or not R6A.get("pesos_sha256"):
    raise SystemExit("ERRO: a fase 6a ainda não terminou. Termine a 6a primeiro ('Executar tudo' no notebook dela).")
BASE = Path(R6A["pasta"]) / "pesos.pt"
if not BASE.is_file():   # o Drive pode estar montado em outro caminho
    BASE = CRIVO / "transformer-crivo-46m" / Path(R6A["pasta"]).name / "pesos.pt"
if sha256(BASE) != R6A["pesos_sha256"]:
    raise SystemExit(f"ERRO: {BASE} não tem o SHA do resumo da 6a ({R6A['pesos_sha256'][:8]}…).")
BASE_SHA = R6A["pesos_sha256"]
SAIDA = CRIVO / "transformer-crivo-46m-conversa"
SAIDA.mkdir(parents=True, exist_ok=True)
DADOS_DRIVE = CRIVO / "dados-fase6b"
DADOS_DRIVE.mkdir(parents=True, exist_ok=True)
LOCAL = Path("/content/fase6b")
LOCAL.mkdir(parents=True, exist_ok=True)
dispositivo = torch.device("cuda")
SEED = 20261015
print("CRIVO:", CRIVO, "| base 6a:", BASE, BASE_SHA[:12], flush=True)


In [ ]:
# Célula 1 — código do CRIVO
REPO = Path("/content/CRIVO")
if REPO.exists():
    shutil.rmtree(REPO)
subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/affectionate-carson-22yoe0",
                "https://github.com/HROSONE/CRIVO", str(REPO)], check=True)
COMMIT = subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip()
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "tokenizers>=0.20,<1"], check=True)
for p in (str(REPO), str(REPO / "scripts"), str(REPO / "experimentos" / "fase6b_conversa_humana"),
          str(REPO / "experimentos" / "fase5_conversa_20261013")):
    if p not in sys.path:
        sys.path.insert(0, p)
import linguagem_profunda as lp
import formato_fase6, gerar_fase6b, cenas_v4
import baixar_fontes_linguagem
import importlib.util
_spec = importlib.util.spec_from_file_location("formato_fase5", REPO / "artefatos" / "conversa_v4f_pt" / "formato_fase5.py")
formato_fase5 = importlib.util.module_from_spec(_spec); _spec.loader.exec_module(formato_fase5)
from tokenizers import Tokenizer
TOK = Tokenizer.from_file(str(BASE.parent / "tokenizer.json"))
ESP = {t: TOK.token_to_id(t) for t in ("<pad>", "<documento>", "<usuario>", "<assistente>", "<fim>")}
DOC, FIM = ESP["<documento>"], ESP["<fim>"]
QUEM = gerar_fase6b.resumo_identidade()
print("commit:", COMMIT, "| quem:", QUEM, flush=True)


In [ ]:
# Célula 2 — dados de conversa (uma vez; depois vêm do Drive) e tokens
DADOS = DADOS_DRIVE / "dados_fase6b.json.gz"
if not DADOS.is_file():
    OASST = LOCAL / "oasst2.messages.jsonl.gz"
    if not OASST.is_file():
        baixar_fontes_linguagem.baixar("oasst2", LOCAL)
    LEGENDAS = gerar_fase6b.LEGENDAS_URL
    subprocess.run([sys.executable, str(REPO / "experimentos/fase6b_conversa_humana/gerar_fase6b.py"), "--oasst", str(OASST),
                    "--legendas", LEGENDAS, "--saida", str(LOCAL / "dados")], cwd=REPO, check=True)
    shutil.copyfile(LOCAL / "dados" / "manifesto_fase6b.json", DADOS_DRIVE / "manifesto_fase6b.json")
    shutil.copyfile(LOCAL / "dados" / "dados_fase6b.json.gz", DADOS)
MANIFESTO_DADOS = json.loads((DADOS_DRIVE / "manifesto_fase6b.json").read_text())
with gzip.open(DADOS, "rb") as f:
    bruto = f.read()
assert hashlib.sha256(bruto).hexdigest() == MANIFESTO_DADOS["dados_sha256"], "dados_fase6b.json.gz corrompido"
DADOS_CONVERSA = json.loads(bruto); del bruto
print("dados:", json.dumps(MANIFESTO_DADOS["por_origem"], ensure_ascii=False), flush=True)
print("relatório:", json.dumps(MANIFESTO_DADOS["relatorio"], ensure_ascii=False), flush=True)

# tokens: cada texto distinto é codificado uma vez, em lote
_textos = set()
for v in DADOS_CONVERSA.values():
    for e in v:
        _textos.add(formato_fase6.limpar(formato_fase6.serializar_contexto(e.get("ctx"))))
        _textos.update(formato_fase6.limpar(t["texto"]) for t in e["turnos"])
_textos = sorted(_textos)
CACHE = dict(zip(_textos, (e.ids for e in TOK.encode_batch(_textos, add_special_tokens=False))))
del _textos
def codificar(texto):
    ids = CACHE.get(texto)
    return ids if ids is not None else TOK.encode(texto, add_special_tokens=False).ids

CTX = 1024
EX = {}   # (parte, origem) -> lista de (ids uint16, alvo bool)
nao_cabe = Counter()
for parte, v in DADOS_CONVERSA.items():
    for e in v:
        try:
            ids, alvo = formato_fase6.montar(codificar, ESP, e.get("ctx"), e["turnos"], contexto=CTX)
        except ValueError:
            nao_cabe[e["origem"]] += 1
            continue
        EX.setdefault((parte, e["origem"]), []).append((np.asarray(ids, np.uint16), np.asarray(alvo, bool)))
TOKENS_ALVO = {f"{p}/{o}": int(sum(a.sum() for _, a in l)) for (p, o), l in sorted(EX.items())}
print("exemplos:", {f"{p}/{o}": len(l) for (p, o), l in sorted(EX.items())}, "| não couberam:", dict(nao_cabe), flush=True)
print("tokens com perda:", TOKENS_ALVO, flush=True)

# leitura: corpus próprio, uma parte do FineWeb-2 (da 6a) e o IDENTIDADE.md
for n in ("treino.bin", "validacao.bin"):
    if not (LOCAL / n).is_file():
        shutil.copyfile(CRIVO / "dados-17m" / n, LOCAL / n)
PROPRIO = np.memmap(LOCAL / "treino.bin", dtype=np.uint16, mode="r")
PROPRIO_VAL = np.memmap(LOCAL / "validacao.bin", dtype=np.uint16, mode="r")
MAN_WEB = json.loads((CRIVO / "corpus-fineweb2-pt" / "manifesto.json").read_text())
for p in (MAN_WEB["partes"][-1], MAN_WEB["val"]):   # a última parte: a que a 6a leu por último
    if not (LOCAL / p["arquivo"]).is_file() or (LOCAL / p["arquivo"]).stat().st_size != 2 * p["tokens"]:
        shutil.copyfile(CRIVO / "corpus-fineweb2-pt" / p["arquivo"], LOCAL / p["arquivo"])
WEB = np.memmap(LOCAL / MAN_WEB["partes"][-1]["arquivo"], dtype=np.uint16, mode="r")
WEB_VAL = np.memmap(LOCAL / MAN_WEB["val"]["arquivo"], dtype=np.uint16, mode="r")
IDENT = np.asarray([DOC] + TOK.encode(formato_fase6.limpar((REPO / "IDENTIDADE.md").read_text(encoding="utf-8")),
                                      add_special_tokens=False).ids + [FIM], np.uint16)
print(f"leitura: próprio {len(PROPRIO)/1e9:.2f} bi | web {len(WEB)/1e6:.0f}M | IDENTIDADE.md {len(IDENT)} tokens", flush=True)


In [ ]:
# Célula 3 — o plano de treino (determinístico: recomeçar dá os mesmos lotes)
EPOCAS = 4
LEGENDAS_POR_EPOCA = 2.0    # tokens de legenda novos por volta, em relação aos do OASST2
ACERVO_POR_EPOCA = 1.0      # idem, para o acervo (amostra nova a cada volta)
LOTE, CONVERSA_POR_LOTE = 64, 48
L = CTX + 1

def empacotar(exemplos, rng):
    """Junta conversas em sequências de CTX+1 tokens; o resto da sequência é <fim> sem perda."""
    ordem = rng.permutation(len(exemplos))
    seqs, alvos, ids_cur, alvo_cur = [], [], [], []
    def fechar():
        ids = np.concatenate(ids_cur); alvo = np.concatenate(alvo_cur)
        s = np.full(L, FIM, np.uint16); a = np.zeros(L, bool)
        s[:len(ids)] = ids; a[:len(alvo)] = alvo
        seqs.append(s); alvos.append(a)
    n = 0
    for i in ordem:
        ids, alvo = exemplos[i]
        if n + len(ids) > L and ids_cur:
            fechar(); ids_cur, alvo_cur, n = [], [], 0
        ids_cur.append(ids); alvo_cur.append(alvo); n += len(ids)
    if ids_cur:
        fechar()
    return seqs, alvos

def amostra_por_tokens(lista, tokens, rng, inicio=0):
    """Pega exemplos de `lista` (já embaralhada) a partir de `inicio` até somar `tokens`."""
    out, soma, i = [], 0, inicio
    while soma < tokens and i < len(lista):
        out.append(lista[i]); soma += len(lista[i][0]); i += 1
    return out, i

rng_plano = np.random.default_rng(SEED)
oasst = EX.get(("treino", "oasst2"), [])
acervo = EX.get(("treino", "acervo"), [])
legendas = EX.get(("treino", "legendas"), [])
acervo = [acervo[i] for i in rng_plano.permutation(len(acervo))]
legendas = [legendas[i] for i in rng_plano.permutation(len(legendas))]
T_OASST = max(sum(len(i) for i, _ in oasst), 200_000)   # piso: com OASST2 pequeno, o acervo e as legendas ainda entram
SEQ, ALVO, pos_leg, pos_ac, por_epoca = [], [], 0, 0, []
for ep in range(EPOCAS):
    rng = np.random.default_rng(SEED + 1 + ep)
    ac, pos_ac = amostra_por_tokens(acervo, ACERVO_POR_EPOCA * T_OASST, rng, pos_ac)
    if pos_ac >= len(acervo):
        pos_ac = 0
    leg, pos_leg = amostra_por_tokens(legendas, LEGENDAS_POR_EPOCA * T_OASST, rng, pos_leg)
    s, a = empacotar(oasst + ac + leg, rng)
    n = len(s) - len(s) % CONVERSA_POR_LOTE
    SEQ += s[:n]; ALVO += a[:n]
    por_epoca.append({"oasst2": len(oasst), "acervo": len(ac), "legendas": len(leg), "sequencias": n})
SEQ, ALVO = np.stack(SEQ), np.stack(ALVO)
PASSOS = len(SEQ) // CONVERSA_POR_LOTE
LEGENDAS_INEDITAS = pos_leg >= len(legendas)
print("por volta:", por_epoca, flush=True)
print(f"PASSOS: {PASSOS} ({len(SEQ)} sequências de conversa + {LOTE - CONVERSA_POR_LOTE} de leitura por passo) | "
      f"{100 * ALVO.mean():.0f}% dos tokens de conversa têm perda", flush=True)
if PASSOS < 50:
    raise SystemExit("ERRO: poucos dados de conversa. Mande o print para o assistente.")

def val_pacotes(parte, origem, max_ex=600):
    lista = EX.get((parte, origem), [])[:max_ex]
    if not lista:
        return None
    s, a = empacotar(lista, np.random.default_rng(SEED))
    return np.stack(s), np.stack(a)
VAL = {f"{p}/{o}": val_pacotes(p, o) for p in ("val", "teste") for o in ("oasst2", "acervo", "legendas")}
VAL = {k: v for k, v in VAL.items() if v is not None}
SELECAO = "val/oasst2" if "val/oasst2" in VAL else next(k for k in VAL if k.startswith("val/"))
print("medidas separadas:", {k: len(v[0]) for k, v in VAL.items()}, "| escolha da melhor versão por", SELECAO, flush=True)


In [ ]:
# Célula 4 — o CRIVO da 6a, ou o checkpoint
CKPT = SAIDA / "checkpoint.pt"
salvo = torch.load(BASE, map_location="cpu", weights_only=True)
CFG = dict(salvo["config"], dropout=0.0)
assert CFG["contexto"] >= CTX, CFG
modelo = lp.LinguagemProfunda(lp.Configuracao(**CFG)).to(dispositivo)
modelo.load_state_dict(salvo["modelo"]); del salvo
print(f"base 6a: {CFG['camadas']} camadas, {sum(p.numel() for p in modelo.parameters())/1e6:.1f}M parâmetros, contexto {CFG['contexto']}", flush=True)
PASSO0, HIST, MELHOR_PERDA, MELHOR_ESTADO, MELHOR_PASSO = 0, [], float("inf"), None, 0
if CKPT.is_file():
    c = torch.load(CKPT, map_location="cpu", weights_only=False)
    if c.get("base_sha256") != BASE_SHA or c.get("passos") != PASSOS:
        raise SystemExit("ERRO: o checkpoint é de outra base ou de outro plano de dados. Apague checkpoint.pt para recomeçar.")
    modelo.load_state_dict(c["modelo"]); PASSO0, HIST, MELHOR_PERDA = c["passo"], c["historico"], c["melhor_perda"]
    MELHOR_ESTADO, MELHOR_PASSO = c["melhor_modelo"], c["melhor_passo"]
    print("continuando do checkpoint, passo", PASSO0, flush=True)


In [ ]:
# Célula 5 — treinar: 48 sequências de conversa + 16 de leitura por passo
LR, LR_MIN, AQUECIMENTO = 1e-4, 1e-5, 100
PESO_LEITURA = 0.5
MEDIR_A_CADA = max(50, PASSOS // 20)
SALVAR_A_CADA, MAX_HORAS = 500, 11.5   # cada checkpoint substituído vai para a lixeira do Drive (e ocupa espaço)

params = list(modelo.parameters())
otim = torch.optim.AdamW([{"params": [p for p in params if p.dim() >= 2], "weight_decay": 0.1},
                          {"params": [p for p in params if p.dim() < 2], "weight_decay": 0.0}], lr=LR, betas=(0.9, 0.95), fused=True)
if CKPT.is_file():
    otim.load_state_dict(c["otim"]); del c
escala = torch.amp.GradScaler("cuda", enabled=(DTYPE == torch.float16))
try:
    rodar = torch.compile(modelo)
except Exception as erro:
    print("sem torch.compile:", erro, flush=True); rodar = modelo
taxa = lambda p: LR * (p + 1) / AQUECIMENTO if p < AQUECIMENTO else LR_MIN + (LR - LR_MIN) * 0.5 * (1 + math.cos(math.pi * min(1.0, (p - AQUECIMENTO) / max(1, PASSOS - AQUECIMENTO))))

def alvos_de(seq, alvo):
    """x = seq[:-1]; y = seq[1:] onde o token tem perda, -100 no resto."""
    s = torch.from_numpy(seq.astype(np.int64))
    y = s[:, 1:].clone(); y[~torch.from_numpy(alvo[:, 1:])] = -100
    return s[:, :-1], y

def leitura(rng, n):
    out = []
    for _ in range(n):
        u = rng.random()
        if u < 0.03:      # o IDENTIDADE.md, inteiro ou um pedaço
            ini = int(rng.integers(0, max(1, len(IDENT) - L)))
            pedaco = IDENT[ini:ini + L]
            s = np.full(L, FIM, np.uint16); s[:len(pedaco)] = pedaco
            a = np.zeros(L, bool); a[1:len(pedaco)] = True
        else:
            fonte = WEB if u < 0.55 else PROPRIO
            i = int(rng.integers(0, len(fonte) - L))
            s = np.asarray(fonte[i:i + L]); a = np.ones(L, bool)
        out.append((s, a))
    return np.stack([s for s, _ in out]), np.stack([a for _, a in out])

def lote(passo):
    i = (passo - 1) * CONVERSA_POR_LOTE
    xc, yc = alvos_de(SEQ[i:i + CONVERSA_POR_LOTE], ALVO[i:i + CONVERSA_POR_LOTE])
    xl, yl = alvos_de(*leitura(np.random.default_rng(SEED * 7 + passo), LOTE - CONVERSA_POR_LOTE))
    return xc.to(dispositivo), yc.to(dispositivo), xl.to(dispositivo), yl.to(dispositivo)

@torch.no_grad()
def perda_pacotes(seq, alvo, tam=16):
    modelo.eval(); soma, n = 0.0, 0
    for j in range(0, len(seq), tam):
        x, y = alvos_de(seq[j:j + tam], alvo[j:j + tam])
        x, y = x.to(dispositivo), y.to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            lg = modelo(x)[0]
        soma += F.cross_entropy(lg.float().reshape(-1, lg.shape[-1]), y.reshape(-1), ignore_index=-100, reduction="sum").item()
        n += int((y != -100).sum())
    return soma / max(1, n)

@torch.no_grad()
def perda_leitura(fonte, ctx, lotes=15, tam=16):
    modelo.eval(); g = np.random.default_rng(SEED); r = []
    for _ in range(lotes):
        ini = g.integers(0, len(fonte) - ctx - 1, size=tam)
        b = torch.from_numpy(np.stack([fonte[i:i + ctx + 1] for i in ini]).astype(np.int64)).to(dispositivo)
        with torch.autocast(device_type="cuda", dtype=DTYPE):
            r.append(modelo(b[:, :-1], b[:, 1:])[1].item())
    return float(np.mean(r))

def medir(passo, separadas=("val",)):
    m = {"passo": passo}
    for k, (s, a) in VAL.items():
        if k.split("/")[0] in separadas:
            m["perda_" + k] = perda_pacotes(s, a)
    m["leitura_proprio_256"] = perda_leitura(PROPRIO_VAL, 256)
    m["leitura_web_1024"] = perda_leitura(WEB_VAL, CTX)
    print(f"  MEDIDA passo {passo}: " + " | ".join(f"{k} {v:.4f}" for k, v in m.items() if k != "passo"), flush=True)
    return m

def salvar(passo):
    tmp = SAIDA / "checkpoint.tmp"
    torch.save({"modelo": modelo.state_dict(), "otim": otim.state_dict(), "passo": passo, "historico": HIST,
                "melhor_perda": MELHOR_PERDA, "melhor_modelo": MELHOR_ESTADO, "melhor_passo": MELHOR_PASSO, "config": CFG, "commit": COMMIT, "base_sha256": BASE_SHA, "passos": PASSOS}, tmp)
    tmp.replace(CKPT)

if PASSO0 == 0:
    HIST.append(medir(0, ("val", "teste")))   # a base da 6a, antes de conversar
t0, passo = time.time(), PASSO0
for passo in range(PASSO0 + 1, PASSOS + 1):
    modelo.train()
    for g in otim.param_groups:
        g["lr"] = taxa(passo)
    xc, yc, xl, yl = lote(passo)
    with torch.autocast(device_type="cuda", dtype=DTYPE):
        lg = rodar(torch.cat([xc, xl]))[0].float()
    V = lg.shape[-1]
    l_conv = F.cross_entropy(lg[:len(xc)].reshape(-1, V), yc.reshape(-1), ignore_index=-100)
    l_leit = F.cross_entropy(lg[len(xc):].reshape(-1, V), yl.reshape(-1), ignore_index=-100)
    l = l_conv + PESO_LEITURA * l_leit
    otim.zero_grad(set_to_none=True)
    escala.scale(l).backward(); escala.unscale_(otim)
    torch.nn.utils.clip_grad_norm_(params, 1.0)
    escala.step(otim); escala.update()
    del lg
    if passo % 50 == 0:
        print(f"passo {passo}/{PASSOS} conversa {l_conv.item():.3f} leitura {l_leit.item():.3f} lr {taxa(passo):.2e} | "
              f"{(time.time()-t0)/max(1,passo-PASSO0):.2f}s/passo", flush=True)
    if passo % MEDIR_A_CADA == 0 or passo == PASSOS:
        HIST.append(medir(passo))
        if HIST[-1]["perda_" + SELECAO] < MELHOR_PERDA:
            MELHOR_PERDA = HIST[-1]["perda_" + SELECAO]
            MELHOR_ESTADO = {k: v.detach().cpu().clone() for k, v in modelo.state_dict().items()}   # na memória, não no Drive
            MELHOR_PASSO = passo
            print(f"  ↑ melhor até agora ({SELECAO} {MELHOR_PERDA:.4f})", flush=True)
    fim_tempo = time.time() - t0 > MAX_HORAS * 3600
    if passo % SALVAR_A_CADA == 0 or passo == PASSOS or fim_tempo:
        salvar(passo)
        print(f"  checkpoint salvo (passo {passo})", flush=True)
    if fim_tempo:
        print("Limite de tempo da sessão: rode 'Executar tudo' de novo para continuar.", flush=True)
        break
TERMINOU = passo >= PASSOS


In [ ]:
# Célula 6 — a melhor versão: medidas finais e o que ele diz (sem retoque)
if not TERMINOU:
    raise SystemExit("Ainda não terminou: rode 'Executar tudo' de novo para continuar.")
modelo.load_state_dict(MELHOR_ESTADO); PASSO_MELHOR = MELHOR_PASSO
FINAL = medir(PASSO_MELHOR, ("val", "teste"))
PROIBIDOS = [ESP["<pad>"], DOC, ESP["<usuario>"], ESP["<assistente>"]]

def falar(ctx, turnos, temperatura=0.0, semente=0, max_tokens=120):
    ids, _ = formato_fase6.montar(codificar, ESP, ctx, turnos, contexto=CTX - max_tokens, gerar=True)
    saida, terminou = modelo.gerar(ids, FIM, max_tokens=max_tokens, temperatura=temperatura, top_k=40, semente=semente,
                                   proibidos=PROIBIDOS)
    return TOK.decode(saida).strip(), terminou

def turnos_de(historico, mensagem):
    t = [{"papel": "usuario" if i % 2 == 0 else "assistente", "texto": h} for i, h in enumerate(historico)]
    return t + [{"papel": "usuario", "texto": mensagem}]

def conferir(ctx, historico, mensagem, texto, terminou, exigir=None):
    caso = {"contexto": {k: v for k, v in (ctx or {}).items() if k in formato_fase5.CAMPOS}, "historico": list(historico),
            "mensagem": mensagem + " " + " ".join((ctx or {}).get("lembrancas", [])) + " " + (ctx or {}).get("quem", ""),
            "exigir": exigir or {}}
    r = formato_fase5.avaliar_saida(caso, texto, terminou)
    return {k: v for k, v in r.items() if k in ("tudo_ok", "usa_fato", "inventados", "sem_repeticao", "terminou")}

FALAS = []
def ouvir(grupo, ctx, historico, mensagem, exigir=None, espera=None):
    for temp, sem in ((0.0, 0), (0.7, 1)):
        texto, terminou = falar(ctx, turnos_de(historico, mensagem), temp, sem)
        FALAS.append({"grupo": grupo, "temperatura": temp, "historico": list(historico), "mensagem": mensagem,
                      "lembrancas": (ctx or {}).get("lembrancas"), "fatos": (ctx or {}).get("fatos"), "resposta": texto,
                      "terminou": terminou, "conferencia": conferir(ctx, historico, mensagem, texto, terminou, exigir),
                      "lembrou": (espera.casefold() in texto.casefold()) if espera else None})

Q = {"quem": QUEM}
for c in cenas_v4.CONVERSAS_REAIS:
    ouvir("frases_reais_do_criador", Q, c["historico"], c["mensagem"])
for m in ("quem é você?", "qual é o seu nome?", "você é uma pessoa?", "o que você mais gosta de fazer?", "você tem sentimentos?",
          "do que você é feito?"):
    ouvir("identidade", Q, [], m)
for m in ("me dá uma dica pra estudar melhor", "como faço arroz?", "por que o céu é azul?", "o que é mais importante na vida?",
          "tô cansado do trabalho", "me conta uma curiosidade", "qual a diferença entre vírus e bactéria?", "você gosta de música?"):
    ouvir("perguntas_livres", Q, [], m)
for lembr, msg, espera in ((["a pessoa disse: meu nome é Rafaela"], "você lembra o meu nome?", "Rafaela"),
                           (["a pessoa disse: tenho um gato chamado Bolinha"], "como se chama o meu gato?", "Bolinha"),
                           (["a pessoa disse: eu moro em Curitiba"], "onde eu moro mesmo?", "Curitiba"),
                           (["a pessoa disse: amanhã tenho prova de química"], "lembra o que eu tenho amanhã?", "química"),
                           (["a pessoa disse: meu time é o Bahia"], "qual é o meu time?", "Bahia"),
                           (["a pessoa disse: minha filha se chama Lúcia", "a pessoa disse: ela tem 7 anos"], "quantos anos tem a minha filha?", "7")):
    ouvir("lembrancas", {"quem": QUEM, "lembrancas": lembr}, [], msg, espera=espera)
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "acervo" and sum(f["grupo"] == "fatos_do_acervo" for f in FALAS) < 40:
        ouvir("fatos_do_acervo", e["ctx"], [], e["turnos"][0]["texto"], exigir={"fato": True})
for e in DADOS_CONVERSA.get("teste", []):
    if e["origem"] == "oasst2" and sum(f["grupo"] == "oasst2_separado" for f in FALAS) < 30:
        ts = e["turnos"][:-1]
        ouvir("oasst2_separado", e["ctx"], [t["texto"] for t in ts[:-1]], ts[-1]["texto"])

def taxa_de(grupo, chave):
    v = [f["conferencia"].get(chave) if chave != "lembrou" else f["lembrou"] for f in FALAS if f["grupo"] == grupo and f["temperatura"] == 0.0]
    v = [x for x in v if x is not None]
    return round(sum(v) / len(v), 3) if v else None
TAXAS = {"conferencia_tudo_ok": {g: taxa_de(g, "tudo_ok") for g in sorted({f["grupo"] for f in FALAS})},
         "usa_fato_acervo": taxa_de("fatos_do_acervo", "usa_fato"), "lembrou": taxa_de("lembrancas", "lembrou")}
for f in FALAS:
    if f["temperatura"] == 0.0 and f["grupo"] in ("frases_reais_do_criador", "identidade", "lembrancas", "perguntas_livres"):
        print(f"[{f['grupo']}] {f['mensagem']!r}\n   → {f['resposta']}", flush=True)
print(json.dumps(TAXAS, ensure_ascii=False, indent=1), flush=True)


In [ ]:
# Célula 7 — salvar a versão que conversa e o resumo pequeno
stamp = datetime.now().strftime("%Y%m%d_%H%M%S")
DEST = SAIDA / f"conversa_{stamp}"
DEST.mkdir(parents=True)
torch.save({"versao": lp.VERSAO, "modelo": {k: v.detach().cpu() for k, v in modelo.state_dict().items()}, "config": CFG,
            "passo": PASSO_MELHOR, "execucao": {"tokenizer_sha256": sha256(BASE.parent / "tokenizer.json"), "pesos_externos": False,
                                                 "base_6a_sha256": BASE_SHA, "protocolo": "fase6b", "commit": COMMIT,
                                                 "dados_sha256": MANIFESTO_DADOS["dados_sha256"]}}, DEST / "pesos.pt")
shutil.copyfile(BASE.parent / "tokenizer.json", DEST / "tokenizer.json")
shutil.copyfile(REPO / "experimentos/fase6b_conversa_humana/formato_fase6.py", DEST / "formato_fase6.py")
resumo = {"protocolo": "fase6b_conversa_humana", "criado_em": datetime.now().isoformat(), "commit": COMMIT, "gpu": GPU,
          "base_6a_sha256": BASE_SHA, "config": CFG, "passos": PASSOS, "passo_escolhido": PASSO_MELHOR, "selecao": SELECAO,
          "epocas": EPOCAS, "por_epoca": por_epoca, "legendas_acabaram": LEGENDAS_INEDITAS, "dados": MANIFESTO_DADOS,
          "tokens_com_perda": TOKENS_ALVO, "nao_couberam": dict(nao_cabe), "antes": HIST[0], "final": FINAL, "historico": HIST,
          "taxas": TAXAS, "falas": FALAS, "pasta": str(DEST), "pesos_sha256": sha256(DEST / "pesos.pt"),
          "nenhuma_fala_escrita_pelo_assistente": True, "ativo_no_chat": False}
(SAIDA / "resumo_fase6b.json").write_text(json.dumps(resumo, ensure_ascii=False, indent=1, default=float))
print("SALVO EM", DEST, "| sha", resumo["pesos_sha256"], flush=True)
print("Avise o assistente: ele lê o resumo_fase6b.json.", flush=True)
